In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isovalericacid,102.13,3.89246,3.4166876,259.9018025,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isovalericacid,H,isovalericacid,e,2000.0,0.03
"""

model = PCSAFT(["isovalericacid"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isovalericacid.csv")
fix_line_endings("rhol_isovalericacid.csv")

Fixed: satp_isovalericacid.csv
Fixed: rhol_isovalericacid.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isovalericacid.csv",
        "satp_isovalericacid.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.0844539494013496


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([3.8314811479246007, 268.9077825033863, 3.446364076571081, 2825.192541383539, 0.0031013678109875075], PCSAFT{BasicIdeal, Float64}("isovalericacid"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isovalericacid.csv",   my_saturation_p)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: satp_isovalericacid.csv ===
Clapeyron Estimator  exp           calc          ARD%    
396.0500    14560.000000  14621.149731  0.4200  
402.9000    19590.000000  19615.677458  0.1311  
412.8700    29650.000000  29335.547281  1.0605  
420.7000    39720.000000  39465.045655  0.6419  
427.1800    49780.000000  49842.143733  0.1248  
432.6200    59860.000000  60154.745041  0.4924  
437.4400    69920.000000  70654.983722  1.0512  
441.6000    79990.000000  80843.884016  1.0675  
445.1600    90070.000000  90456.264022  0.4288  
446.6000    95100.000000  94591.011076  0.5352  
447.9200    100080.000000  98510.468256  1.5683  
AARD = 0.6838%


0.6837927946436566

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isovalericacid.csv",   my_liquid_density)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: rhol_isovalericacid.csv ===
Clapeyron Estimator  exp           calc          ARD%    
273.1500    945.000000    941.655466    0.3539  
293.1400    927.000000    925.043942    0.2110  
313.1300    909.100000    908.505886    0.0654  
333.1200    891.100000    891.784200    0.0768  
353.1200    873.200000    874.651471    0.1662  
373.1200    854.800000    856.931532    0.2494  
393.1300    836.200000    838.460342    0.2703  
413.1300    817.400000    819.125044    0.2110  
433.1400    798.000000    798.788893    0.0989  
453.1400    777.200000    777.355712    0.0200  
473.1500    755.000000    754.670513    0.0436  
493.1600    731.000000    730.569824    0.0588  
513.1700    705.700000    704.808812    0.1263  
533.1700    678.500000    677.043434    0.2147  
AARD = 0.1547%


0.1547384853852056